# Diccionarios · Más colecciones

Más práctica con diccionarios y con el módulo `collections` aplicada a problemas concretos: vistas de claves como conjuntos, inventarios con `Counter`, agrupaciones con `defaultdict`, el orden en `OrderedDict`, colas y turnos con `deque`, registros con `namedtuple` y un diccionario propio con `UserDict`. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios) y [PYnative](https://pynative.com/python-exercises-with-solutions/).

## 1. Valores únicos y listas más largas

Un club de senderismo guarda en un diccionario `ruta → lista de inscritos`. Escribir una función que diga si todos los valores de un diccionario son distintos (sirve también con listas, que no son *hashables*), ordenar las rutas por número de inscritos de mayor a menor y devolver todas las rutas empatadas con la lista más larga.

In [1]:
import json
import random
from collections import Counter, OrderedDict, UserDict, defaultdict, deque, namedtuple

inscritos = {
    "Roque Nublo": ["Ana", "Yeray", "Iballa"],
    "Barranco de Guayadeque": ["Ana", "Acerina"],
    "Pico de la Zarza": ["Tanausú", "Iballa", "Dácil", "Ancor"],
    "Caldera de Bandama": ["Yeray"],
    "Roque del Conde": ["Acerina", "Ancor", "Ana", "Yeray"],
}


def valores_unicos(d):
    def hashable(v):
        return tuple(v) if isinstance(v, list) else v

    return len({hashable(v) for v in d.values()}) == len(d)


def mas_largas(d):
    maximo = max(map(len, d.values()))
    return [k for k, v in d.items() if len(v) == maximo]


por_inscritos = dict(sorted(inscritos.items(), key=lambda kv: len(kv[1]), reverse=True))
for ruta, personas in por_inscritos.items():
    print(f"{ruta:<24} {len(personas)}  {', '.join(personas)}")
print("Más concurridas:", mas_largas(inscritos))
print("Valores únicos:", valores_unicos(inscritos), valores_unicos({"a": 1, "b": 2, "c": 1}))

assert mas_largas(inscritos) == ["Pico de la Zarza", "Roque del Conde"]
assert max(inscritos, key=lambda k: len(inscritos[k])) == "Pico de la Zarza"  # max solo da el primero
assert not valores_unicos({"x": [1, 2], "y": [1, 2]})
assert [len(v) for v in por_inscritos.values()] == sorted(map(len, inscritos.values()), reverse=True)

Pico de la Zarza         4  Tanausú, Iballa, Dácil, Ancor
Roque del Conde          4  Acerina, Ancor, Ana, Yeray
Roque Nublo              3  Ana, Yeray, Iballa
Barranco de Guayadeque   2  Ana, Acerina
Caldera de Bandama       1  Yeray
Más concurridas: ['Pico de la Zarza', 'Roque del Conde']
Valores únicos: True False


## 2. Comparar dos diccionarios con sus vistas

Dos fruterías publican sus precios por kilo. Usando que `keys()` e `items()` se comportan como conjuntos, obtener los productos que venden las dos, los que vende solo una de ellas, los que venden a la vez con el mismo precio y, de los comunes, en cuál es más barato cada uno. Comprobar cada resultado con una comprensión que no use operaciones de conjuntos.

In [2]:
mercado = {"plátano": 1.35, "papaya": 2.80, "aguacate": 4.50, "mango": 3.90, "naranja": 1.20}
barrio = {"plátano": 1.35, "aguacate": 4.20, "naranja": 1.40, "níspero": 3.10, "papa": 1.60}

comunes = mercado.keys() & barrio.keys()
solo_mercado = mercado.keys() - barrio.keys()
solo_barrio = barrio.keys() - mercado.keys()
en_una_sola = mercado.keys() ^ barrio.keys()
mismo_precio = dict(mercado.items() & barrio.items())

print("Comunes:", sorted(comunes))
print("Solo en el mercado:", sorted(solo_mercado), "| solo en el barrio:", sorted(solo_barrio))
print("Mismo precio:", mismo_precio)
for producto in sorted(comunes):
    a, b = mercado[producto], barrio[producto]
    donde = "igual" if a == b else "mercado" if a < b else "barrio"
    print(f"  {producto:<9} {a:>5.2f} {b:>5.2f}  → {donde}")

assert comunes == {k for k in mercado if k in barrio}
assert en_una_sola == solo_mercado | solo_barrio == {k for k in [*mercado, *barrio] if (k in mercado) != (k in barrio)}
assert mismo_precio == {k: v for k, v in mercado.items() if barrio.get(k) == v}

Comunes: ['aguacate', 'naranja', 'plátano']
Solo en el mercado: ['mango', 'papaya'] | solo en el barrio: ['níspero', 'papa']
Mismo precio: {'plátano': 1.35}
  aguacate   4.50  4.20  → barrio
  naranja    1.20  1.40  → mercado
  plátano    1.35  1.35  → igual


## 3. Inventario con `Counter`

Un almacén lleva sus existencias en un `Counter`. Sumar un envío con `+`, restar un pedido con `-` (que descarta lo que queda a cero o en negativo) y repetirlo con `subtract`, que conserva los negativos y sirve para saber cuántas unidades faltan para servir el pedido completo. Mostrar los tres productos con más existencias y el total de unidades.

In [3]:
almacen = Counter(gofio=40, mojo=25, queso=12, vino=30, millo=8)
envio = Counter(mojo=10, queso=6, almogrote=15)
pedido = Counter(gofio=20, queso=20, millo=8, vino=5)

tras_envio = almacen + envio
tras_pedido = tras_envio - pedido
print("Tras el envío:", dict(tras_envio))
print("Tras el pedido (-):", dict(tras_pedido))

con_negativos = tras_envio.copy()
con_negativos.subtract(pedido)
faltan = -con_negativos  # el menos unario se queda con los negativos, cambiados de signo
print("Con subtract:", dict(con_negativos))
print("Faltan:", dict(faltan))
print("Más existencias:", tras_pedido.most_common(3), "| total:", tras_pedido.total())

assert tras_pedido == +con_negativos  # el más unario descarta ceros y negativos
assert faltan == Counter(queso=2)
assert "millo" not in tras_pedido and con_negativos["millo"] == 0
assert tras_pedido.total() == sum(v for v in con_negativos.values() if v > 0)

Tras el envío: {'gofio': 40, 'mojo': 35, 'queso': 18, 'vino': 30, 'millo': 8, 'almogrote': 15}
Tras el pedido (-): {'gofio': 20, 'mojo': 35, 'vino': 25, 'almogrote': 15}
Con subtract: {'gofio': 20, 'mojo': 35, 'queso': -2, 'vino': 25, 'millo': 0, 'almogrote': 15}
Faltan: {'queso': 2}
Más existencias: [('mojo', 35), ('vino', 25), ('gofio', 20)] | total: 95


## 4. Agrupar con `defaultdict`

- Con los registros `(sede, departamento, empleado)`, construir un diccionario anidado `sede → departamento → lista de empleados` con `defaultdict` y convertirlo a diccionarios normales para mostrarlo.
- Con las matrículas `(alumno, asignatura)`, que tienen duplicados, obtener las asignaturas distintas de cada alumno con `defaultdict(set)`.
- Con pares dirigidos `(origen, destino)`, construir la lista de adyacencia de un grafo, incluidos los nodos sin salidas, y el grado de entrada de cada nodo.

Comprobar cada resultado con una versión hecha con `setdefault` y ver por qué hay que consultar un `defaultdict` con `in` o `get` y no con corchetes.

In [4]:
registros = [
    ("Las Palmas", "ventas", "Ana"), ("Santa Cruz", "ventas", "Yeray"), ("Las Palmas", "soporte", "Iballa"),
    ("Las Palmas", "ventas", "Ancor"), ("Santa Cruz", "almacén", "Dácil"), ("Santa Cruz", "ventas", "Tanausú"),
]

plantilla = defaultdict(lambda: defaultdict(list))
for sede, depto, empleado in registros:
    plantilla[sede][depto].append(empleado)
plantilla = {sede: dict(deptos) for sede, deptos in plantilla.items()}
print(json.dumps(plantilla, ensure_ascii=False, indent=2))

manual = {}
for sede, depto, empleado in registros:
    manual.setdefault(sede, {}).setdefault(depto, []).append(empleado)
assert plantilla == manual

{
  "Las Palmas": {
    "ventas": [
      "Ana",
      "Ancor"
    ],
    "soporte": [
      "Iballa"
    ]
  },
  "Santa Cruz": {
    "ventas": [
      "Yeray",
      "Tanausú"
    ],
    "almacén": [
      "Dácil"
    ]
  }
}


In [5]:
matriculas = [("Ana", "Física"), ("Ana", "Química"), ("Yeray", "Física"), ("Ana", "Física"), ("Yeray", "Dibujo"),
              ("Yeray", "Física"), ("Iballa", "Química")]
asignaturas = defaultdict(set)
for alumno, asignatura in matriculas:
    asignaturas[alumno].add(asignatura)
for alumno, conjunto in asignaturas.items():
    print(f"{alumno:<7} {sorted(conjunto)}")
assert asignaturas == {a: {s for b, s in matriculas if b == a} for a, _ in matriculas}

print("¿Está Dácil?", "Dácil" in asignaturas, "| get:", asignaturas.get("Dácil"))
_ = asignaturas["Dácil"]  # consultar con corchetes crea la clave
print("Tras asignaturas['Dácil']:", "Dácil" in asignaturas, asignaturas["Dácil"])

Ana     ['Física', 'Química']
Yeray   ['Dibujo', 'Física']
Iballa  ['Química']
¿Está Dácil? False | get: None
Tras asignaturas['Dácil']: True set()


In [6]:
aristas = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E"), ("F", "C")]
adyacencia = defaultdict(list)
entrada = Counter()
for origen, destino in aristas:
    adyacencia[origen].append(destino)
    adyacencia[destino]  # para que los nodos sin salidas también aparezcan
    entrada[destino] += 1
for nodo in sorted(adyacencia):
    print(f"{nodo} → {adyacencia[nodo]!s:<12} entrada: {entrada[nodo]}")

assert sorted(adyacencia) == sorted({n for arista in aristas for n in arista})
assert sum(map(len, adyacencia.values())) == entrada.total() == len(aristas)
assert [n for n in sorted(adyacencia) if entrada[n] == 0] == ["A", "F"]

A → ['B', 'C']   entrada: 0
B → ['D']        entrada: 1
C → ['D']        entrada: 2
D → ['E']        entrada: 2
E → []           entrada: 1
F → ['C']        entrada: 0


## 5. El orden en `OrderedDict`

Una cola de embarque se guarda en un `OrderedDict` `pasajero → asiento`. Mover a un pasajero prioritario al principio con `move_to_end(last=False)` y a uno que llega tarde al final, embarcar en orden con `popitem(last=False)` y recorrer la cola al revés. Comprobar que dos `OrderedDict` con los mismos pares en distinto orden son distintos, pero que un `OrderedDict` y un `dict` se comparan sin tener en cuenta el orden.

In [7]:
cola = OrderedDict([("Ana", "12A"), ("Yeray", "3C"), ("Iballa", "21F"), ("Ancor", "8B"), ("Dácil", "1A")])
cola.move_to_end("Dácil", last=False)
cola.move_to_end("Yeray")
print("Cola:", list(cola))
print("Al revés:", list(reversed(cola)))

embarcados = []
while len(cola) > 2:
    embarcados.append(cola.popitem(last=False))
print("Embarcados:", embarcados, "| quedan:", dict(cola))
assert [p for p, _ in embarcados] == ["Dácil", "Ana", "Iballa"]
assert list(cola) == ["Ancor", "Yeray"]

Cola: ['Dácil', 'Ana', 'Iballa', 'Ancor', 'Yeray']
Al revés: ['Yeray', 'Ancor', 'Iballa', 'Ana', 'Dácil']
Embarcados: [('Dácil', '1A'), ('Ana', '12A'), ('Iballa', '21F')] | quedan: {'Ancor': '8B', 'Yeray': '3C'}


In [8]:
a = OrderedDict(x=1, y=2)
b = OrderedDict(y=2, x=1)
print(f"{'OrderedDict == OrderedDict':<28} {a == b}")
print(f"{'OrderedDict == dict':<28} {a == dict(b)}")
print(f"{'dict == dict':<28} {dict(a) == dict(b)}")
assert a != b and a == dict(b) and dict(a) == dict(b)

OrderedDict == OrderedDict   False
OrderedDict == dict          True
dict == dict                 True


## 6. Colas y turnos con `deque`

- Operar por los dos extremos de una `deque` y ver que `extendleft` mete los elementos en orden inverso.
- Atender a los clientes de un puesto del mercado en orden de llegada (cola FIFO).
- Repartir el procesador entre tareas por turnos (*round-robin*): cada tarea trabaja como mucho 3 unidades de tiempo y, si no ha terminado, vuelve al final de la cola. Calcular cuándo termina cada una. Usar `rotate` para pasar el turno en un juego de mesa.
- Guardar las 5 últimas búsquedas con `maxlen=5`.

In [9]:
d = deque([3, 4])
d.append(5)
d.appendleft(2)
d.extendleft([1, 0])  # mete 1 y luego 0: quedan al revés
d.extend([6, 7])
print("Antes:", list(d))
print("pop:", d.pop(), "| popleft:", d.popleft(), "| después:", list(d))
assert list(d) == [1, 2, 3, 4, 5, 6]

clientes = deque()
for nombre in ["Ana", "Yeray", "Iballa"]:
    clientes.append(nombre)
atendidos = [clientes.popleft()]
clientes.append("Ancor")
while clientes:
    atendidos.append(clientes.popleft())
print("Orden de atención:", atendidos)
assert atendidos == ["Ana", "Yeray", "Iballa", "Ancor"]

Antes: [0, 1, 2, 3, 4, 5, 6, 7]
pop: 7 | popleft: 0 | después: [1, 2, 3, 4, 5, 6]
Orden de atención: ['Ana', 'Yeray', 'Iballa', 'Ancor']


In [10]:
def round_robin(tareas, quantum):
    cola, reloj, fin = deque(tareas.items()), 0, {}
    while cola:
        nombre, restante = cola.popleft()
        trabajo = min(quantum, restante)
        reloj += trabajo
        if restante > trabajo:
            cola.append((nombre, restante - trabajo))
        else:
            fin[nombre] = reloj
    return fin


def round_robin_paso_a_paso(tareas, quantum):
    # Simula unidad a unidad con una lista: más lento pero evidente
    pendientes, reloj, fin = [[n, t] for n, t in tareas.items()], 0, {}
    while pendientes:
        tarea = pendientes.pop(0)
        for _ in range(quantum):
            reloj += 1
            tarea[1] -= 1
            if tarea[1] == 0:
                fin[tarea[0]] = reloj
                break
        else:
            pendientes.append(tarea)
    return fin


tareas = {"copia": 5, "correo": 2, "informe": 7, "imprimir": 3}
fin = round_robin(tareas, quantum=3)
for nombre, t in fin.items():
    print(f"{nombre:<9} dura {tareas[nombre]} y termina en t={t}")
assert fin == round_robin_paso_a_paso(tareas, 3)
assert max(fin.values()) == sum(tareas.values())

random.seed(5)
for _ in range(200):
    azar = {f"t{i}": random.randint(1, 9) for i in range(random.randint(1, 6))}
    q = random.randint(1, 4)
    assert round_robin(azar, q) == round_robin_paso_a_paso(azar, q)

correo    dura 2 y termina en t=5
imprimir  dura 3 y termina en t=11
copia     dura 5 y termina en t=13
informe   dura 7 y termina en t=17


In [11]:
jugadores = deque(["Ana", "Yeray", "Iballa", "Ancor"])
turnos = []
for _ in range(6):
    turnos.append(jugadores[0])
    jugadores.rotate(-1)  # el que ha jugado pasa al final
print("Turnos:", turnos)
lista = ["Ana", "Yeray", "Iballa", "Ancor"]
for k in range(-6, 7):
    d = deque(lista)
    d.rotate(k)
    assert list(d) == lista[-k % 4:] + lista[:-k % 4]

busquedas = deque(maxlen=5)
for termino in ["guagua", "ferry", "playa", "tiempo", "mareas", "sendero", "fiestas"]:
    busquedas.append(termino)
print("Últimas búsquedas:", list(busquedas))
assert list(busquedas) == ["playa", "tiempo", "mareas", "sendero", "fiestas"]

Turnos: ['Ana', 'Yeray', 'Iballa', 'Ancor', 'Ana', 'Yeray']
Últimas búsquedas: ['playa', 'tiempo', 'mareas', 'sendero', 'fiestas']


## 7. Historial de un navegador

Simular los botones *atrás* y *adelante* de un navegador con dos pilas: al ir atrás, la página actual pasa a la pila de adelante, y al visitar una página nueva la pila de adelante se vacía. Si no se puede ir atrás o adelante, la página no cambia. Ejecutar una secuencia fija de acciones mostrando el estado tras cada una y comprobarlo con otra implementación: una lista de páginas y un índice.

In [12]:
class Navegador:
    def __init__(self, inicio):
        self.actual, self.atras, self.adelante = inicio, [], []

    def visitar(self, pagina):
        self.atras.append(self.actual)
        self.actual = pagina
        self.adelante.clear()

    def ir_atras(self):
        if self.atras:
            self.adelante.append(self.actual)
            self.actual = self.atras.pop()

    def ir_adelante(self):
        if self.adelante:
            self.atras.append(self.actual)
            self.actual = self.adelante.pop()


class NavegadorLista:
    def __init__(self, inicio):
        self.paginas, self.i = [inicio], 0

    @property
    def actual(self):
        return self.paginas[self.i]

    def visitar(self, pagina):
        del self.paginas[self.i + 1:]
        self.paginas.append(pagina)
        self.i += 1

    def ir_atras(self):
        self.i = max(self.i - 1, 0)

    def ir_adelante(self):
        self.i = min(self.i + 1, len(self.paginas) - 1)


acciones = ["noticias", "deportes", "<", "<", ">", "tiempo", ">", "<", "<", "<", ">", ">"]
pilas, lista = Navegador("inicio"), NavegadorLista("inicio")
print(f"{'acción':<10} {'actual':<9} {'atrás':<30} adelante")
for accion in acciones:
    for nav in (pilas, lista):
        match accion:
            case "<": nav.ir_atras()
            case ">": nav.ir_adelante()
            case pagina: nav.visitar(pagina)
    print(f"{accion:<10} {pilas.actual:<9} {str(pilas.atras):<30} {pilas.adelante}")
    assert pilas.actual == lista.actual
    assert pilas.atras == lista.paginas[:lista.i] and pilas.adelante[::-1] == lista.paginas[lista.i + 1:]
assert pilas.actual == "tiempo" and pilas.adelante == []

acción     actual    atrás                          adelante
noticias   noticias  ['inicio']                     []
deportes   deportes  ['inicio', 'noticias']         []
<          noticias  ['inicio']                     ['deportes']
<          inicio    []                             ['deportes', 'noticias']
>          noticias  ['inicio']                     ['deportes']
tiempo     tiempo    ['inicio', 'noticias']         []
>          tiempo    ['inicio', 'noticias']         []
<          noticias  ['inicio']                     ['tiempo']
<          inicio    []                             ['tiempo', 'noticias']
<          inicio    []                             ['tiempo', 'noticias']
>          noticias  ['inicio']                     ['tiempo']
>          tiempo    ['inicio', 'noticias']         []


## 8. Registros con `namedtuple`

Leer las paradas de una línea de guaguas desde un texto CSV y convertir cada fila en una `namedtuple` `Parada` con `_make`, usando `_fields` para comprobar la cabecera. El campo `accesible` vale `True` por defecto (`defaults=`): comprobar que el constructor lo usa pero `_make` no. Cambiar una parada con `_replace` (sin modificar la original), pasar las paradas a JSON con `_asdict` y reconstruirlas desde el JSON.

In [13]:
import csv
import io

Parada = namedtuple("Parada", "codigo nombre minuto accesible", defaults=(True,))
texto_csv = """codigo,nombre,minuto,accesible
101,Intercambiador,0,sí
102,Plaza de España,4,sí
103,Mesa y López,7,no
104,Auditorio,12,sí
"""

filas = csv.reader(io.StringIO(texto_csv))
cabecera = next(filas)
assert tuple(cabecera) == Parada._fields
paradas = [Parada._make([int(c), n, int(m), a == "sí"]) for c, n, m, a in filas]
print(Parada._fields, Parada._field_defaults)
print(Parada(105, "Teatro", 15))
try:
    Parada._make([105, "Teatro", 15])
except TypeError as e:
    print("_make no usa los valores por defecto:", e)
for p in paradas:
    print(f"{p.codigo} {p.nombre:<16} min {p.minuto:>2}  accesible: {'sí' if p.accesible else 'no'}")

retrasada = paradas[3]._replace(minuto=paradas[3].minuto + 5)
print(paradas[3].minuto, "→", retrasada.minuto)
assert paradas[3].minuto == 12 and retrasada.minuto == 17

datos = json.dumps([p._asdict() for p in paradas], ensure_ascii=False)
print(datos[:90] + "…")
recuperadas = [Parada(**d) for d in json.loads(datos)]
assert recuperadas == paradas
assert Parada(1, "x", 0) == (1, "x", 0, True)  # sigue siendo una tupla

('codigo', 'nombre', 'minuto', 'accesible') {'accesible': True}
Parada(codigo=105, nombre='Teatro', minuto=15, accesible=True)
_make no usa los valores por defecto: Expected 4 arguments, got 3
101 Intercambiador   min  0  accesible: sí
102 Plaza de España  min  4  accesible: sí
103 Mesa y López     min  7  accesible: no
104 Auditorio        min 12  accesible: sí
12 → 17
[{"codigo": 101, "nombre": "Intercambiador", "minuto": 0, "accesible": true}, {"codigo": 1…


## 9. Tabla de efectividad entre elementos

En un juego, los ataques de un elemento son más o menos eficaces según el elemento del defensor. Guardar la tabla como un diccionario de diccionarios con solo los casos distintos de 1 (`2` muy eficaz, `0.5` poco eficaz) y escribir `efectividad(ataque, defensa)` con `get` en dos niveles. Calcular el daño con la fórmula propia $\text{daño} = \operatorname{round}\left(\frac{\text{potencia} \cdot \text{ataque}}{2 \cdot \text{defensa}} \cdot \text{efectividad}\right)$, rechazando potencias fuera de 1-100 y elementos desconocidos. Mostrar la tabla completa.

In [14]:
ELEMENTOS = ["fuego", "agua", "planta", "eléctrico"]
TABLA = {
    "fuego": {"fuego": 0.5, "agua": 0.5, "planta": 2},
    "agua": {"fuego": 2, "agua": 0.5, "planta": 0.5},
    "planta": {"fuego": 0.5, "agua": 2, "planta": 0.5},
    "eléctrico": {"agua": 2, "planta": 0.5, "eléctrico": 0.5},
}
MENSAJES = {2: "¡Es muy eficaz!", 0.5: "No es muy eficaz…", 1: ""}


def efectividad(ataque, defensa):
    if ataque not in ELEMENTOS or defensa not in ELEMENTOS:
        raise ValueError(f"elemento desconocido: {ataque!r} o {defensa!r}")
    return TABLA.get(ataque, {}).get(defensa, 1)


def danio(elem_ataque, elem_defensa, potencia, ataque, defensa):
    if not 1 <= potencia <= 100:
        raise ValueError(f"potencia fuera de 1-100: {potencia}")
    return round(potencia * ataque / (2 * defensa) * efectividad(elem_ataque, elem_defensa))


print(f"{'ataca ↓ / defiende →':<22}" + "".join(f"{e:>10}" for e in ELEMENTOS))
for a in ELEMENTOS:
    print(f"{a:<22}" + "".join(f"{efectividad(a, d):>10g}" for d in ELEMENTOS))

ataca ↓ / defiende →       fuego      agua    planta eléctrico
fuego                        0.5       0.5         2         1
agua                           2       0.5       0.5         1
planta                       0.5         2       0.5         1
eléctrico                      1         2       0.5       0.5


In [15]:
combates = [("agua", "fuego", 60, 80, 50), ("fuego", "agua", 60, 80, 50), ("eléctrico", "fuego", 60, 80, 50)]
for e_at, e_def, potencia, at, de in combates:
    print(f"{e_at:>9} → {e_def:<6} daño {danio(e_at, e_def, potencia, at, de):>3}  {MENSAJES[efectividad(e_at, e_def)]}")

for caso in [("agua", "fuego", 150, 80, 50), ("hielo", "agua", 50, 80, 50)]:
    try:
        danio(*caso)
    except ValueError as e:
        print("Error:", e)

assert danio("agua", "fuego", 60, 80, 50) == 4 * danio("fuego", "agua", 60, 80, 50) == 96
assert all(efectividad(e, e) == 0.5 for e in ELEMENTOS)
assert all(v in (0.5, 2) for fila in TABLA.values() for v in fila.values())

     agua → fuego  daño  96  ¡Es muy eficaz!
    fuego → agua   daño  24  No es muy eficaz…
eléctrico → fuego  daño  48  
Error: potencia fuera de 1-100: 150
Error: elemento desconocido: 'hielo' o 'agua'


## 10. Un diccionario propio con `UserDict`

Crear `SinTildes`, un diccionario cuyas claves de texto no distinguen mayúsculas ni tildes (`'Gáldar'`, `'galdar'` y `'GALDAR'` son la misma clave), heredando de `collections.UserDict` y redefiniendo solo `__setitem__`, `__getitem__`, `__delitem__` y `__contains__`. Comprobar que el constructor, `update`, `get`, `in`, `setdefault` y `pop` respetan la normalización, y que no pasa lo mismo si se hereda de `dict`, cuyos métodos no llaman a los que se redefinen. (`ChainMap` ya aparece en *Diccionarios · Avanzado*.)

In [16]:
SIN_TILDES = str.maketrans("áéíóúüÁÉÍÓÚÜ", "aeiouuAEIOUU")


def normalizar(clave):
    return clave.translate(SIN_TILDES).casefold() if isinstance(clave, str) else clave


class SinTildes(UserDict):
    def __setitem__(self, clave, valor):
        super().__setitem__(normalizar(clave), valor)

    def __getitem__(self, clave):
        return super().__getitem__(normalizar(clave))

    def __delitem__(self, clave):
        super().__delitem__(normalizar(clave))

    def __contains__(self, clave):
        return normalizar(clave) in self.data


class SinTildesDict(dict):
    def __setitem__(self, clave, valor):
        super().__setitem__(normalizar(clave), valor)

    def __getitem__(self, clave):
        return super().__getitem__(normalizar(clave))


inicial = {"Gáldar": "Gran Canaria", "Güímar": "Tenerife", "Pájara": "Fuerteventura"}
bueno, malo = SinTildes(inicial), SinTildesDict(inicial)
for d in (bueno, malo):
    d.update({"TÍAS": "Lanzarote"})
    d["Valverde"] = "El Hierro"

pruebas = {
    "d['GALDAR']": lambda d: d["GALDAR"],
    "'guimar' in d": lambda d: "guimar" in d,
    "d.get('pajara')": lambda d: d.get("pajara"),
    "d.get('tias')": lambda d: d.get("tias"),
    "'VALVERDE' in d": lambda d: "VALVERDE" in d,
}
print(f"{'operación':<18} {'UserDict':<15} dict")
for nombre, prueba in pruebas.items():
    resultados = []
    for d in (bueno, malo):
        try:
            resultados.append(repr(prueba(d)))
        except KeyError as e:
            resultados.append(f"KeyError({e})")
    print(f"{nombre:<18} {resultados[0]:<15} {resultados[1]}")

assert list(bueno) == ["galdar", "guimar", "pajara", "tias", "valverde"]
assert bueno.setdefault("Pájara", "?") == "Fuerteventura" and bueno.pop("GÜÍMAR") == "Tenerife"
assert "Güímar" not in bueno and len(bueno) == 4
assert "Gáldar" in malo and "galdar" not in malo  # el constructor de dict no pasa por __setitem__

operación          UserDict        dict
d['GALDAR']        'Gran Canaria'  KeyError('galdar')
'guimar' in d      True            False
d.get('pajara')    'Fuerteventura' None
d.get('tias')      'Lanzarote'     None
'VALVERDE' in d    True            False
